# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the "Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review" dataset, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json) and contains tables for clinical features, hormone levels and imaging, and CYP21A2 genetic variants.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using mlcroissant.

The dataset source is defined by a Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")
print("\nPublished:", getattr(metadata, 'datePublished', ''))
print("License:", getattr(metadata, 'license', ''))
print("\nKeywords:", getattr(metadata, 'keywords', ''))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate all record sets, fields and columns using their `@id` for systematic extraction and downstream processing.

In [ ]:
# List all record sets by their @id
record_sets = []
for rs in dataset.metadata.recordSet:
    print(f"Record Set: {rs['@id']} (name: {rs.get('name', '---')})")
    record_sets.append(rs['@id'])
    # List all fields in the record set
    fields = rs.get('field', [])
    print("  Fields:")
    for fld in fields:
        print(f"    {fld['@id']} (name: {fld.get('name', '---')}, type: {fld.get('dataType', '---')})")
    print()
# Optionally, list columns in each file object (if present)
for rs in dataset.metadata.recordSet:
    file_objs = rs.get('fileObject', [])
    if file_objs:
        for fo in file_objs:
            columns = fo.get('column', [])
            print(f"Columns in FileObject {fo['@id']}:")
            for col in columns:
                print(f"  Column: {col['@id']} (name: {col.get('name', '---')}, type: {col.get('dataType', '---')})")
            print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references to entities (record sets, fields) use their `@id`.

Below we extract each record set using its `@id` and store them as pandas DataFrames.

In [ ]:
# Extract records from each record set
# Use @id for referencing

dfs = {}
for rs_id in record_sets:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dfs[rs_id] = pd.DataFrame(records)
        print(f"Columns: {dfs[rs_id].columns.tolist()}")
        display(dfs[rs_id].head())
    else:
        print("No records found.")
    print()
# Choose one record set for downstream EDA. If the first record set exists, use it for demonstration:
if record_sets:
    selected_record_set_id = record_sets[0]
    print(f'Using {selected_record_set_id} for EDA.')
    df = dfs[selected_record_set_id]
else:
    selected_record_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

All operations are done using the `@id`s for fields and columns. Let's select a numeric field for demonstration and perform filtering and normalization.

In [ ]:
if df is not None:
    # Identify a numeric field (@id) from your record set metadata
    numeric_field_id = None
    group_field_id = None
    # Try to select the first numeric field
    for rs in dataset.metadata.recordSet:
        if rs['@id'] == selected_record_set_id:
            for fld in rs.get('field', []):
                dt = fld.get('dataType', '').lower()
                if dt in ['schema:integer', 'schema:float', 'integer', 'float', 'number']:  # Common Croissant types
                    numeric_field_id = fld['@id']
                    break
            # Try to select a group field (@id) - use a text/categorical field
            for fld in rs.get('field', []):
                dt = fld.get('dataType', '').lower()
                if dt == 'schema:text':
                    group_field_id = fld['@id']
                    break
            break

    # If not found, fall back to the first column name
    if not numeric_field_id and not df.empty:
        for col in df.columns:
            # try to pick an integer/float column
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not group_field_id and not df.empty:
        # Try to pick a text column
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]):
                group_field_id = col
                break
    
    print(f"Numeric field for analysis: {numeric_field_id}")
    print(f"Group field for analysis: {group_field_id}")
    
    # Filtering: Set arbitrary threshold for demonstration
    threshold = df[numeric_field_id].mean() if numeric_field_id else None
    if numeric_field_id and threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We use matplotlib to display simple distributions. Customization can use available numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated stepwise exploration of a clinical and genomic dataset using `mlcroissant`, referencing all entities via their `@id` for reproducible analysis. You can extend these steps for more detailed comparisons, combine tables, or integrate domain-specific analysis.

Key findings and summary statistics can be further extracted for research or clinical purposes.

*Notebook complete. For more advanced tasks, see the [`mlcroissant`](https://github.com/mlcommons/croissant) documentation or extend preprocessing and visualization as needed.*